In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from bertopic import BERTopic

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.decomposition import LatentDirichletAllocation
from gensim.models import CoherenceModel
#from bertopic import BERTopic
import pyLDAvis
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
import re
import warnings
from gensim.corpora import Dictionary
import joblib
import json
import itertools
from gensim.models.coherencemodel import CoherenceModel

In [ ]:
data = pd.read_csv('/content/drive/MyDrive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/train.csv')
data.head()

In [ ]:
# Unsupervised Yöntem deneneceğinden konu sütunları ve gereksiz sütunlar silinir.
data_no_topic = data.drop(['ID','Computer Science', 'Physics', 'Mathematics','Statistics', 'Quantitative Biology', 'Quantitative Finance'], axis=1)
data_no_topic.head()

In [ ]:
# Konu ve Özet sütunları birleştirilir.
data_no_topic['combined_text'] = data_no_topic['TITLE'] + " " + data_no_topic['ABSTRACT']
data_no_topic.head()

In [ ]:
#Boş değer kontrolü yapılır.
data_no_topic.isnull().any()

In [ ]:
# Gerekli yüklemeler yapılır.
import nltk
nltk.download('punkt', download_dir='/content/drive/MyDrive/nltk_data/')
nltk.download('stopwords', download_dir='/content/drive/MyDrive/nltk_data/')
nltk.download('wordnet', download_dir='/content/drive/MyDrive/nltk_data/')
nltk.data.path.append('/content/drive/MyDrive/nltk_data/')
nltk.download('punkt_tab')

In [ ]:
# Stopwords ve lemmatizer
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

# Metin temizleme fonksiyonu
def preprocess_text(text):
    text = re.sub(r'\W', ' ', text)  # Noktalama işaretlerini kaldır
    text = re.sub(r'\s+', ' ', text)  # Fazla boşlukları temizle
    text = text.lower()  # Küçük harfe çevir
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(word) for word in tokens if word not in stop_words and len(word) > 2]
    return ' '.join(tokens)

# Birleştirilmiş metin temizlenir
data_no_topic['cleaned_combined_text'] = data_no_topic['combined_text'].apply(preprocess_text)
data_no_topic.head()

In [ ]:
# Metinler tokenize edilir
tokenized_texts = [text.split() for text in data_no_topic['cleaned_combined_text']]

In [ ]:
# Gensim Dictionary ve Corpus oluşturulur
gensim_dictionary = Dictionary(tokenized_texts)
gensim_corpus = [gensim_dictionary.doc2bow(text) for text in tokenized_texts]

# Gensim Dictionary ve Corpus kaydedilir
gensim_dictionary.save("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/gensim_dictionary.dict")
joblib.dump(gensim_corpus, "/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/gensim_corpus.pkl")
print("Gensim Dictionary ve Corpus kaydedildi.")

In [ ]:
# Gensim Dictionary ve Corpus yüklenir
gensim_dictionary = Dictionary.load("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/gensim_dictionary.dict")
gensim_corpus = joblib.load("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/gensim_corpus.pkl")
print("Gensim Dictionary ve Corpus yüklendi.")

In [ ]:
##########################

In [ ]:
from gensim.corpora import Dictionary
from gensim.models.ldamodel import LdaModel
from gensim.models import CoherenceModel

# Veri hazırlama (gensim formatı)
tokenized_texts = [text.split() for text in data_no_topic['cleaned_combined_text']]
dictionary = Dictionary(tokenized_texts)
corpus = [dictionary.doc2bow(text) for text in tokenized_texts]

# LDA Modeli - Hiçbir konu sınırlandırması olmadan
lda_model = LdaModel(corpus=corpus, id2word=dictionary, random_state=42)

# İlk çıktıyı inceleme
topics = lda_model.print_topics(num_words=10)
for topic in topics:
    print(f"Topic {topic[0]}: {topic[1]}")


In [ ]:
import pyLDAvis.gensim_models as gensimvis
# PyLDAvis için hazırlık
lda_vis = gensimvis.prepare(lda_model, corpus, dictionary)

# Görselleştirme
pyLDAvis.enable_notebook()  # Jupyter Notebook kullanıyorsanız
pyLDAvis.display(lda_vis)

In [ ]:
#LDA için
# İlk 8 konuyu ve en önemli 4 kelimeyi almak için bir fonksiyon
def get_first_8_topics(lda_model, num_words=4):
    top_terms = {}
    for topic_id in range(8):  # İlk 8 konuyu al
        terms = lda_model.show_topic(topic_id, topn=num_words)
        top_terms[f"Topic {topic_id}"] = [(word, round(weight, 4)) for word, weight in terms]
    return top_terms

# İlk 8 konuyu al
first_8_topics = get_first_8_topics(lda_model, num_words=4)


In [ ]:
import matplotlib.pyplot as plt

# Her bir konuyu görselleştirme
def plot_first_8_topics(top_terms):
    num_topics = len(top_terms)
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))  # 2 satır, 4 sütun
    axes = axes.flatten()

    for i, (topic_name, terms) in enumerate(top_terms.items()):
        words = [term[0] for term in terms]
        weights = [term[1] for term in terms]

        # Bar grafiği
        axes[i].barh(words, weights, color="skyblue")
        axes[i].set_title(topic_name, fontsize=14)
        axes[i].invert_yaxis()  # Kelimelerin üstte görünmesi için
        axes[i].set_xlabel("Weight")

        # Y eksenine kelimeleri ekle
        axes[i].set_yticks(range(len(words)))
        axes[i].set_yticklabels(words, fontsize=12)

    # Gereksiz boşlukları kaldır ve grafikleri düzenle
    plt.tight_layout()
    plt.show()

# Görselleştirme
plot_first_8_topics(first_8_topics)


In [ ]:

from bertopic import BERTopic

# ==== BERTopic İŞLEMLERİ ====
# BERTopic modelini oluştur ve uygulama
output_path = "/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje"  # Sonuçların kaydedileceği klasör yolu
bertopic_model = BERTopic(language="english")
topics, probs = bertopic_model.fit_transform(data_no_topic['cleaned_combined_text'])

# BERTopic sonuçlarını kaydetme
topic_info = bertopic_model.get_topic_info()
topic_info.to_csv(f"{output_path}/bertopic_topic_info.csv", index=False)
pd.DataFrame({"topics": topics, "probs": probs}).to_csv(f"{output_path}/bertopic_topics_probs.csv", index=False)

# Görselleştirme (opsiyonel)
bertopic_model.visualize_topics().write_html(f"{output_path}/bertopic_visualization.html")


In [ ]:
from IPython.core.display import display, HTML

# HTML dosyasını oku ve göster
with open("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/bertopic_visualization.html", "r") as file:
    html_content = file.read()

display(HTML(html_content))


In [ ]:
# BERTopic sonuçlarından en önemli kelimeleri çıkar
def get_bertopic_top_terms(model, num_topics=8, num_words=4):
    topics = model.get_topics()
    top_terms = {}
    sorted_topics = sorted(topics.items(), key=lambda x: x[0])  # Konuları sıralı hale getir

    for topic_id, words in sorted_topics[:num_topics]:  # İlk n konuyu al
        if topic_id == -1:  # Eğer "Outlier" konular varsa, atla
            continue
        top_terms[topic_id] = [(word, round(weight, 4)) for word, weight in words[:num_words]]
    return top_terms

# İlk 8 konunun en önemli terimlerini al
#Outlier olduğu için num_topics = 9 belirledim.
bertopic_top_terms = get_bertopic_top_terms(bertopic_model, num_topics=9, num_words=4)


In [ ]:
import matplotlib.pyplot as plt

# Her bir konuyu görselleştirme
def plot_bertopic_top_terms(top_terms):
    num_topics = len(top_terms)
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))  # 2 satır, 4 sütun
    axes = axes.flatten()

    for i, (topic_id, terms) in enumerate(top_terms.items()):
        words = [term[0] for term in terms]
        weights = [term[1] for term in terms]

        # Bar grafiği
        axes[i].barh(words, weights, color="skyblue")
        axes[i].set_title(f"Topic {topic_id}", fontsize=14)
        axes[i].invert_yaxis()  # Kelimelerin üstte görünmesi için
        axes[i].set_xlabel("Weight")

        # Y eksenine kelimeleri ekle
        axes[i].set_yticks(range(len(words)))
        axes[i].set_yticklabels(words, fontsize=12)

    # Gereksiz boşlukları kaldır ve grafikleri düzenle
    plt.tight_layout()
    plt.show()

# Görselleştirme
plot_bertopic_top_terms(bertopic_top_terms)


In [ ]:
#LDA ile konu modelleme ve optimizasyon

In [ ]:
# LDA için perplexity ve coherence hesaplanması
def evaluate_lda_model(lda_model, X, tokenized_texts, feature_names, gensim_dictionary):
    perplexity = lda_model.perplexity(X)
    topics = [[feature_names[i] for i in topic.argsort()[-10:]] for topic in lda_model.components_]

    # Farklı coherence metriklerini hesaplama
    coherence_models = {
        "c_v": CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=gensim_dictionary, coherence='c_v'),
        "u_mass": CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=gensim_dictionary, coherence='u_mass'),
        "uci": CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=gensim_dictionary, coherence='c_uci')
    }

    coherence_scores = {key: model.get_coherence() for key, model in coherence_models.items()}
    return perplexity, coherence_scores

In [ ]:
# Lambda kombinasyonları ile optimal konu sayısını bulma
def find_optimal_topics_with_lambda(X, vectorizer, tokenized_texts, gensim_dictionary, output_path):
    num_topics_range = range(2, 21)  # 2-20 arasında konu sayısı
    feature_names = vectorizer.get_feature_names_out()

    # Sonuçları saklama
    results = []
    for num_topics in num_topics_range:
        lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
        lda.fit(X)
        perplexity, coherence_scores = evaluate_lda_model(
            lda, X, tokenized_texts, feature_names, gensim_dictionary
        )
        results.append({
            "num_topics": num_topics,
            "perplexity": perplexity,
            **coherence_scores
        })

    # Sonuçları DataFrame'e dönüştür
    results_df = pd.DataFrame(results)

    # Normalizasyon
    for col in ["perplexity", "c_v", "u_mass", "uci"]:
        if col == "perplexity" or col == "u_mass":
            results_df[col + "_norm"] = (results_df[col] - results_df[col].max()) / (results_df[col].min() - results_df[col].max())
        else:
            results_df[col + "_norm"] = (results_df[col] - results_df[col].min()) / (results_df[col].max() - results_df[col].min())

    # Lambda kombinasyonlarını oluştur
    lambdas = np.linspace(0, 1, 5)
    lambda_combinations = list(itertools.product(lambdas, repeat=4))

    # Lambda kombinasyonlarını test et
    final_results = []
    def calculate_combined_score(row, lambdas):
        return (
            lambdas[0] * row["perplexity_norm"] +
            lambdas[1] * row["c_v_norm"] +
            lambdas[2] * row["u_mass_norm"] +
            lambdas[3] * row["uci_norm"]
        )

    for combination in lambda_combinations:
        results_df["combined_score"] = results_df.apply(calculate_combined_score, axis=1, lambdas=combination)
        best_row = results_df.loc[results_df["combined_score"].idxmax()]
        final_results.append({
            "lambda": combination,
            "best_num_topics": best_row["num_topics"],
            "best_score": best_row["combined_score"]
        })

    # En iyi sonucu seç
    final_results_df = pd.DataFrame(final_results)
    best_combination = final_results_df.loc[final_results_df["best_score"].idxmax()]
    print(f"En iyi Lambda Kombinasyonu: {best_combination['lambda']}")
    print(f"Optimal Konu Sayısı: {best_combination['best_num_topics']}")

    # Sonuçları kaydet
    results_df.to_csv(f"{output_path}/lda_results_with_normalized_scores.csv", index=False)
    final_results_df.to_csv(f"{output_path}/optimal_lambda_results.csv", index=False)

    return best_combination, final_results_df

In [ ]:
# Bag-of-Words modeli
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
X = vectorizer.fit_transform(data_no_topic['cleaned_combined_text'])

# Vectorizer ve X kaydedilir
from scipy.sparse import save_npz, load_npz
joblib.dump(vectorizer, "/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/vectorizer.pkl")
save_npz("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/X_matrix.npz", X)
print("Vectorizer ve belge-terim matrisi kaydedildi.")

In [ ]:
# Vectorizer ve X yüklenir
from scipy.sparse import save_npz, load_npz
vectorizer = joblib.load("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/vectorizer.pkl")
X = load_npz("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/X_matrix.npz")
print("Vectorizer ve belge-terim matrisi başarıyla yüklendi.")

# Optimum konu sayısı ve lambda kombinasyonları belirlenir
output_path = "/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje"  # Sonuçların kaydedileceği klasör yolu
optimal_combination, all_results = find_optimal_topics_with_lambda(
    X, vectorizer, tokenized_texts, gensim_dictionary, output_path
)

# Sonuçları kaydedilir
results = {
    "optimal_combination": optimal_combination,
    "all_results": all_results
}

In [ ]:
all_results = results["all_results"]
all_results[all_results["best_score"]>3]

In [ ]:
# LDA modeli oluştur ve eğit
best_num_topics = 20  # Optimal konu sayısını buraya koyabilirsiniz
lda = LatentDirichletAllocation(n_components=best_num_topics, random_state=42)
lda.fit(X)

# PyLDAvis verisi hazırlama
def sklearn_lda_to_pyldavis(lda_model, X, vectorizer):
    data = {
        'topic_term_dists': np.array(lda_model.components_) / lda_model.components_.sum(axis=1)[:, np.newaxis],
        'doc_topic_dists': lda_model.transform(X),
        'doc_lengths': np.array(X.sum(axis=1)).flatten(),
        'vocab': vectorizer.get_feature_names_out(),
        'term_frequency': np.array(X.sum(axis=0)).flatten()
    }
    return data

# PyLDAvis verisi hazırlama
data = sklearn_lda_to_pyldavis(lda, X, vectorizer)

# PyLDAvis görselleştirme
vis_data = pyLDAvis.prepare(**data)
pyLDAvis.display(vis_data)


In [ ]:
# BERTopic sonuçlarından en önemli kelimeleri çıkar
def get_bertopic_top_terms(model, num_topics=8, num_words=4):
    topics = model.get_topics()
    top_terms = {}
    sorted_topics = sorted(topics.items(), key=lambda x: x[0])  # Konuları sıralı hale getir

    for topic_id, words in sorted_topics[:num_topics]:  # İlk n konuyu al
        if topic_id == -1:  # Eğer "Outlier" konular varsa, atla
            continue
        top_terms[topic_id] = [(word, round(weight, 4)) for word, weight in words[:num_words]]
    return top_terms

In [ ]:
#LDA için
# İlk 8 konuyu ve en önemli 4 kelimeyi almak için bir fonksiyon
def get_first_8_topics(lda_model, vectorizer, num_words=4):
    top_terms = {}
    feature_names = vectorizer.get_feature_names_out()
    for topic_id, topic_weights in enumerate(lda_model.components_[:8]):
        top_words = [(feature_names[i], topic_weights[i]) for i in topic_weights.argsort()[:-num_words-1:-1]]
        top_terms[f"Topic {topic_id}"] = [(word, round(weight, 4)) for word, weight in top_words]
    return top_terms

# İlk 8 konuyu al
first_8_topics = get_first_8_topics(lda, vectorizer, num_words=4)
print(first_8_topics)

In [ ]:
import matplotlib.pyplot as plt

# Her bir konuyu görselleştirme
def plot_first_8_topics(top_terms):
    num_topics = len(top_terms)
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))  # 2 satır, 4 sütun
    axes = axes.flatten()

    for i, (topic_name, terms) in enumerate(top_terms.items()):
        words = [term[0] for term in terms]
        weights = [term[1] for term in terms]

        # Bar grafiği
        axes[i].barh(words, weights, color="skyblue")
        axes[i].set_title(topic_name, fontsize=14)
        axes[i].invert_yaxis()  # Kelimelerin üstte görünmesi için
        axes[i].set_xlabel("Weight")

        # Y eksenine kelimeleri ekle
        axes[i].set_yticks(range(len(words)))
        axes[i].set_yticklabels(words, fontsize=12)

    # Gereksiz boşlukları kaldır ve grafikleri düzenle
    plt.tight_layout()
    plt.show()

# Görselleştirme
plot_first_8_topics(first_8_topics)

In [ ]:
from bertopic import BERTopic

# BERTopic modelini 16 konu ile oluştur
bertopic_model = BERTopic(language="english", nr_topics=20)

# Modeli veri üzerinde eğit
topics, probs = bertopic_model.fit_transform(data_no_topic['cleaned_combined_text'])

# Distance map görselleştirme
bertopic_model.visualize_topics()

In [ ]:
# BERTopic sonuçlarından en önemli kelimeleri çıkar
def get_bertopic_top_terms(model, num_topics=8, num_words=4):
    topics = model.get_topics()
    top_terms = {}
    sorted_topics = sorted(topics.items(), key=lambda x: x[0])  # Konuları sıralı hale getir

    for topic_id, words in sorted_topics[:num_topics]:  # İlk n konuyu al
        if topic_id == -1:  # Eğer "Outlier" konular varsa, atla
            continue
        top_terms[topic_id] = [(word, round(weight, 4)) for word, weight in words[:num_words]]
    return top_terms

# İlk 8 konunun en önemli terimlerini al
#Outlier olduğu için num_topics = 9 belirledim.
bertopic_top_terms = get_bertopic_top_terms(bertopic_model, num_topics=9, num_words=4)

In [ ]:
# Her bir konuyu görselleştirme
def plot_bertopic_top_terms(top_terms):
    num_topics = len(top_terms)
    fig, axes = plt.subplots(2, 4, figsize=(20, 10))  # 2 satır, 4 sütun
    axes = axes.flatten()

    for i, (topic_id, terms) in enumerate(top_terms.items()):
        words = [term[0] for term in terms]
        weights = [term[1] for term in terms]

        # Bar grafiği
        axes[i].barh(words, weights, color="skyblue")
        axes[i].set_title(f"Topic {topic_id}", fontsize=14)
        axes[i].invert_yaxis()  # Kelimelerin üstte görünmesi için
        axes[i].set_xlabel("Weight")

        # Y eksenine kelimeleri ekle
        axes[i].set_yticks(range(len(words)))
        axes[i].set_yticklabels(words, fontsize=12)

    # Gereksiz boşlukları kaldır ve grafikleri düzenle
    plt.tight_layout()
    plt.show()

# Görselleştirme
plot_bertopic_top_terms(bertopic_top_terms)

In [ ]:
########################################################################################

#### LDA ile Konu Modelleme ve Optimizasyon

In [ ]:
# LDA için perplexity ve coherence hesaplanması
def evaluate_lda_model(lda_model, X, tokenized_texts, feature_names, gensim_dictionary):
    perplexity = lda_model.perplexity(X)
    topics = [[feature_names[i] for i in topic.argsort()[-10:]] for topic in lda_model.components_]

    # Farklı coherence metriklerini hesaplama
    coherence_models = {
        "c_v": CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=gensim_dictionary, coherence='c_v'),
        "u_mass": CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=gensim_dictionary, coherence='u_mass'),
        "uci": CoherenceModel(topics=topics, texts=tokenized_texts, dictionary=gensim_dictionary, coherence='c_uci')
    }

    coherence_scores = {key: model.get_coherence() for key, model in coherence_models.items()}
    return perplexity, coherence_scores

In [ ]:
# Lambda kombinasyonları ile optimal konu sayısını bulma
def find_optimal_topics_with_lambda(X, vectorizer, tokenized_texts, gensim_dictionary, output_path):
    num_topics_range = range(2, 21)  # 2-20 arasında konu sayısı
    feature_names = vectorizer.get_feature_names_out()

    # Sonuçları saklama
    results = []
    for num_topics in num_topics_range:
        lda = LatentDirichletAllocation(n_components=num_topics, random_state=42)
        lda.fit(X)
        perplexity, coherence_scores = evaluate_lda_model(
            lda, X, tokenized_texts, feature_names, gensim_dictionary
        )
        results.append({
            "num_topics": num_topics,
            "perplexity": perplexity,
            **coherence_scores
        })

    # Sonuçları DataFrame'e dönüştür
    results_df = pd.DataFrame(results)

    # Normalizasyon
    for col in ["perplexity", "c_v", "u_mass", "uci"]:
        if col == "perplexity" or col == "u_mass":
            results_df[col + "_norm"] = (results_df[col] - results_df[col].max()) / (results_df[col].min() - results_df[col].max())
        else:
            results_df[col + "_norm"] = (results_df[col] - results_df[col].min()) / (results_df[col].max() - results_df[col].min())

    # Lambda kombinasyonlarını oluştur
    lambdas = np.linspace(0, 1, 5)
    lambda_combinations = list(itertools.product(lambdas, repeat=4))

    # Lambda kombinasyonlarını test et
    final_results = []
    def calculate_combined_score(row, lambdas):
        return (
            lambdas[0] * row["perplexity_norm"] +
            lambdas[1] * row["c_v_norm"] +
            lambdas[2] * row["u_mass_norm"] +
            lambdas[3] * row["uci_norm"]
        )

    for combination in lambda_combinations:
        results_df["combined_score"] = results_df.apply(calculate_combined_score, axis=1, lambdas=combination)
        best_row = results_df.loc[results_df["combined_score"].idxmax()]
        final_results.append({
            "lambda": combination,
            "best_num_topics": best_row["num_topics"],
            "best_score": best_row["combined_score"]
        })

    # En iyi sonucu seç
    final_results_df = pd.DataFrame(final_results)
    best_combination = final_results_df.loc[final_results_df["best_score"].idxmax()]
    print(f"En iyi Lambda Kombinasyonu: {best_combination['lambda']}")
    print(f"Optimal Konu Sayısı: {best_combination['best_num_topics']}")

    # Sonuçları kaydet
    results_df.to_csv(f"{output_path}/lda_results_with_normalized_scores.csv", index=False)
    final_results_df.to_csv(f"{output_path}/optimal_lambda_results.csv", index=False)

    return best_combination, final_results_df

In [ ]:
# Bag-of-Words modeli
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
X = vectorizer.fit_transform(data_no_topic['cleaned_combined_text'])

# Vectorizer ve X kaydedilir
from scipy.sparse import save_npz, load_npz
joblib.dump(vectorizer, "/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/vectorizer.pkl")
save_npz("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/X_matrix.npz", X)
print("Vectorizer ve belge-terim matrisi kaydedildi.")

In [ ]:
# Vectorizer ve X yüklenir
from scipy.sparse import save_npz, load_npz
vectorizer = joblib.load("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/vectorizer.pkl")
X = load_npz("/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje/X_matrix.npz")
print("Vectorizer ve belge-terim matrisi başarıyla yüklendi.")

# Optimum konu sayısı ve lambda kombinasyonları belirlenir
output_path = "/content/drive/My Drive/Yüksek Lisans 2. Dönem/Veri Madenciliği/Proje"  # Sonuçların kaydedileceği klasör yolu
optimal_combination, all_results = find_optimal_topics_with_lambda(
    X, vectorizer, tokenized_texts, gensim_dictionary, output_path
)

# Sonuçları kaydedilir
results = {
    "optimal_combination": optimal_combination,
    "all_results": all_results
}

In [ ]:
all_results.info()

In [ ]:
all_results[all_results["best_score"]>3]

In [ ]:
len(lda.components_[0])

In [ ]:
# LDA modeli oluştur ve eğit
best_num_topics = 5  # Optimal konu sayısını buraya koyabilirsiniz
lda = LatentDirichletAllocation(n_components=best_num_topics, random_state=42)
lda.fit(X)

# PyLDAvis verisi hazırlama
def sklearn_lda_to_pyldavis(lda_model, X, vectorizer):
    data = {
        'topic_term_dists': np.array(lda_model.components_) / lda_model.components_.sum(axis=1)[:, np.newaxis],
        'doc_topic_dists': lda_model.transform(X),
        'doc_lengths': np.array(X.sum(axis=1)).flatten(),
        'vocab': vectorizer.get_feature_names_out(),
        'term_frequency': np.array(X.sum(axis=0)).flatten()
    }
    return data

# PyLDAvis verisi hazırlama
data = sklearn_lda_to_pyldavis(lda, X, vectorizer)

# PyLDAvis görselleştirme
vis_data = pyLDAvis.prepare(**data)
pyLDAvis.display(vis_data)

### BERT

In [ ]:
import numpy as np
import pandas as pd
import pyLDAvis
from bertopic import BERTopic
from sklearn.feature_extraction.text import CountVectorizer

# BERTopic modeli oluştur ve eğit
bertopic_model = BERTopic(language="english")
topics, probs = bertopic_model.fit_transform(data_no_topic['cleaned_combined_text'])

# PyLDAvis verisi hazırlama
def sklearn_bertopic_to_pyldavis(bertopic_model, vectorizer, data_no_topic):
    X = vectorizer.fit_transform(data_no_topic['cleaned_combined_text'])
    vocab = vectorizer.get_feature_names_out()
    term_frequency = np.array(X.sum(axis=0)).flatten()

    # Belge uzunluklarını hesapla
    doc_lengths = np.array([len(doc.split()) for doc in data_no_topic['cleaned_combined_text']])

    # Konu-Kelime ve Belge-Konu dağılımları
    topic_term_dists = np.array([np.array([1/len(vocab) for _ in range(len(vocab))]) for _ in range(len(set(topics)))]).T
    doc_topic_dists = np.array(probs)

    data = {
        'topic_term_dists': topic_term_dists,
        'doc_topic_dists': doc_topic_dists,
        'doc_lengths': doc_lengths,
        'vocab': vocab,
        'term_frequency': term_frequency
    }
    return data

# CountVectorizer ile kelime bilgisi çıkarımı
vectorizer = CountVectorizer(max_df=0.95, min_df=2, stop_words='english')
data = sklearn_bertopic_to_pyldavis(bertopic_model, vectorizer, data_no_topic)

# PyLDAvis görselleştirme
vis_data = pyLDAvis.prepare(**data)
pyLDAvis.display(vis_data)
